# ObjectMemory convolutional SNN — clean experiment notebook

This notebook contains only the current best pipeline and the planned output-threshold experiment.

Run cells from top to bottom. Dataset generation is cached after the first successful run. The baseline and the two threshold experiments are separate at the end, so you can run only the comparison you need.

Required packages: `torch`, `snntorch`, `pandas`, and `numpy`. Keep `lif_utils.py` in the same folder as this notebook.

In [15]:
from pathlib import Path
import copy
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import snntorch as snn
from snntorch import surrogate


def find_support_file(filename):
    candidates = []
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([root / filename, root / "src" / filename])
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Keep it beside this notebook or under src/."
    )


lif_utils_path = find_support_file("lif_utils.py")
sys.path.insert(0, str(lif_utils_path.parent))
import lif_utils as lu

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)
print("Using lif_utils:", lif_utils_path)

Using device: mps
Using lif_utils: /Users/alexeysolganik/Documents/PhD/Summer26/snn-moving-digit/notebooks/lif_utils.py


In [2]:
SHAPE_TO_LABEL = {"seven": 0, "box": 1, "tee": 2, "ell": 3}
LABEL_TO_SHAPE = {label: name for name, label in SHAPE_TO_LABEL.items()}
SHAPE_NAMES = list(SHAPE_TO_LABEL)

GOOD_VELOCITY_PARAMS = {
    "threshold_velocity": 15.0,
    "input_scale": 1.0,
    "beta_velocity": 0.8,
    "inhibition_strength": 3.0,
    "inhibition_radius": 1,
    "internal_steps": 5,
}

CANVAS_HEIGHT = 64
CANVAS_WIDTH = 64
STEPS = 7
INTENSITY = 1.0
SHAPE_TEMPLATES = lu.make_toy_2d_shape_templates()

DIRECTION_CONFIG = {
    "right": {"velocity_y": 0, "velocity_x": 1, "start_top": 28, "start_left": 10},
    "left":  {"velocity_y": 0, "velocity_x": -1, "start_top": 28, "start_left": 45},
    "down":  {"velocity_y": 1, "velocity_x": 0, "start_top": 10, "start_left": 28},
    "up":    {"velocity_y": -1, "velocity_x": 0, "start_top": 45, "start_left": 28},
}

# The original helper module expects these experiment constants as globals.
lu.GOOD_VELOCITY_PARAMS = GOOD_VELOCITY_PARAMS
lu.CANVAS_HEIGHT = CANVAS_HEIGHT
lu.CANVAS_WIDTH = CANVAS_WIDTH
lu.STEPS = STEPS
lu.INTENSITY = INTENSITY
lu.SHAPE_TEMPLATES = SHAPE_TEMPLATES
lu.DIRECTION_CONFIG = DIRECTION_CONFIG

In [3]:
def sample_random_velocity_sequence(
    num_transitions, max_speed=3, allow_stationary=False, seed=None
):
    if seed is not None:
        torch.manual_seed(seed)

    velocities = []
    for _ in range(num_transitions):
        while True:
            dy = int(torch.randint(-max_speed, max_speed + 1, (1,)).item())
            dx = int(torch.randint(-max_speed, max_speed + 1, (1,)).item())
            if allow_stationary or dy != 0 or dx != 0:
                break
        velocities.append((dy, dx))
    return velocities


def choose_safe_start_position_for_velocity_sequence(
    shape_name,
    velocity_sequence,
    canvas_height=CANVAS_HEIGHT,
    canvas_width=CANVAS_WIDTH,
    margin=8,
    seed=None,
):
    if seed is not None:
        torch.manual_seed(seed)

    shape_height, shape_width = SHAPE_TEMPLATES[shape_name].shape
    cumulative_y = [0]
    cumulative_x = [0]
    current_y = current_x = 0

    for dy, dx in velocity_sequence:
        current_y += dy
        current_x += dx
        cumulative_y.append(current_y)
        cumulative_x.append(current_x)

    min_start_y = margin - min(cumulative_y)
    max_start_y = canvas_height - shape_height - margin - max(cumulative_y)
    min_start_x = margin - min(cumulative_x)
    max_start_x = canvas_width - shape_width - margin - max(cumulative_x)

    if min_start_y > max_start_y or min_start_x > max_start_x:
        raise ValueError("Velocity sequence is too large to fit safely on the canvas.")

    start_top = int(torch.randint(min_start_y, max_start_y + 1, (1,)).item())
    start_left = int(torch.randint(min_start_x, max_start_x + 1, (1,)).item())
    return start_top, start_left


def make_one_classifier_dataset_sample(
    shape_name,
    max_speed=3,
    sample_seed=0,
    max_velocity=3,
    max_displacement=18,
    velocity_params=None,
):
    if velocity_params is None:
        velocity_params = GOOD_VELOCITY_PARAMS

    velocity_sequence = sample_random_velocity_sequence(
        STEPS - 1, max_speed=max_speed, seed=sample_seed
    )
    start_top, start_left = choose_safe_start_position_for_velocity_sequence(
        shape_name, velocity_sequence, seed=sample_seed + 10_000
    )
    case = lu.make_variable_velocity_spike_case(
        velocity_sequence=velocity_sequence,
        shape_name=shape_name,
        start_top=start_top,
        start_left=start_left,
    )
    memory_result = lu.run_displacement_gated_object_memory(
        case=case,
        max_velocity=max_velocity,
        max_displacement=max_displacement,
        velocity_params=velocity_params,
        threshold_memory=1.5,
        beta_memory=0.8,
        reset_mode="subtract",
    )
    return {
        "raw_spikes": case["spikes"].float(),
        "object_memory_spikes": memory_result["memory_spike_history"].float(),
        "label": SHAPE_TO_LABEL[shape_name],
    }


def build_classifier_dataset(
    samples_per_shape=100,
    max_speed=3,
    max_velocity=3,
    max_displacement=18,
    base_seed=0,
):
    raw_samples, memory_samples, labels = [], [], []
    total = samples_per_shape * len(SHAPE_NAMES)
    sample_index = 0

    for shape_name in SHAPE_NAMES:
        for _ in range(samples_per_shape):
            sample = make_one_classifier_dataset_sample(
                shape_name=shape_name,
                max_speed=max_speed,
                sample_seed=base_seed + sample_index,
                max_velocity=max_velocity,
                max_displacement=max_displacement,
            )
            raw_samples.append(sample["raw_spikes"])
            memory_samples.append(sample["object_memory_spikes"])
            labels.append(sample["label"])
            sample_index += 1
            if sample_index % 25 == 0 or sample_index == total:
                print(f"Generated {sample_index}/{total} samples")

    return {
        "raw_spikes_dataset": torch.stack(raw_samples),
        "object_memory_spikes_dataset": torch.stack(memory_samples),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

## Build or load the dataset

The first run generates 400 samples and may take a while. It then saves `object_memory_classifier_dataset.pt` beside the notebook. Future runs load that cache quickly.

Delete the cache only when changing the data-generation or ObjectMemory parameters.

In [4]:
CACHE_PATH = lif_utils_path.parent / "object_memory_classifier_dataset.pt"

if CACHE_PATH.exists():
    classifier_dataset = torch.load(CACHE_PATH, map_location="cpu", weights_only=True)
    print("Loaded cached dataset:", CACHE_PATH)
else:
    classifier_dataset = build_classifier_dataset(
        samples_per_shape=100,
        max_speed=3,
        max_velocity=3,
        max_displacement=18,
        base_seed=0,
    )
    torch.save(classifier_dataset, CACHE_PATH)
    print("Saved dataset cache:", CACHE_PATH)

raw_spikes_dataset = classifier_dataset["raw_spikes_dataset"]
object_memory_spikes_dataset = classifier_dataset["object_memory_spikes_dataset"]
labels = classifier_dataset["labels"]

num_classes = len(SHAPE_TO_LABEL)
height, width = raw_spikes_dataset.shape[2:]

print("Raw:", tuple(raw_spikes_dataset.shape))
print("ObjectMemory:", tuple(object_memory_spikes_dataset.shape))
print("Labels:", tuple(labels.shape))
print("Class counts:", torch.bincount(labels).tolist())

Generated 25/400 samples
Generated 50/400 samples
Generated 75/400 samples
Generated 100/400 samples
Generated 125/400 samples
Generated 150/400 samples
Generated 175/400 samples
Generated 200/400 samples
Generated 225/400 samples
Generated 250/400 samples
Generated 275/400 samples
Generated 300/400 samples
Generated 325/400 samples
Generated 350/400 samples
Generated 375/400 samples
Generated 400/400 samples
Saved dataset cache: /Users/alexeysolganik/Documents/PhD/Summer26/snn-moving-digit/src/object_memory_classifier_dataset.pt
Raw: (400, 7, 64, 64)
ObjectMemory: (400, 7, 64, 64)
Labels: (400,)
Class counts: [100, 100, 100, 100]


In [5]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_stratified_train_test_split(labels, train_fraction=0.8, seed=0):
    generator = torch.Generator().manual_seed(seed)
    train_indices, test_indices = [], []

    for class_label in torch.unique(labels):
        class_indices = torch.nonzero(labels == class_label, as_tuple=False).squeeze(1)
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        train_size = int(train_fraction * len(class_indices))
        train_indices.append(class_indices[:train_size])
        test_indices.append(class_indices[train_size:])

    train_indices = torch.cat(train_indices)
    test_indices = torch.cat(test_indices)
    train_indices = train_indices[torch.randperm(len(train_indices), generator=generator)]
    test_indices = test_indices[torch.randperm(len(test_indices), generator=generator)]
    return train_indices, test_indices


def make_dataloaders(spikes, labels, train_indices, test_indices, batch_size=32):
    train_dataset = TensorDataset(
        spikes[train_indices].float(), labels[train_indices].long()
    )
    test_dataset = TensorDataset(
        spikes[test_indices].float(), labels[test_indices].long()
    )
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
        DataLoader(test_dataset, batch_size=batch_size, shuffle=False),
    )


train_indices, test_indices = make_stratified_train_test_split(labels, seed=0)
raw_train_loader, raw_test_loader = make_dataloaders(
    raw_spikes_dataset, labels, train_indices, test_indices
)
memory_train_loader, memory_test_loader = make_dataloaders(
    object_memory_spikes_dataset, labels, train_indices, test_indices
)

print("Train samples:", len(train_indices))
print("Test samples:", len(test_indices))
print("Test class counts:", torch.bincount(labels[test_indices]).tolist())

Train samples: 320
Test samples: 80
Test class counts: [20, 20, 20, 20]


In [6]:
class ConvSpikingClassifier(nn.Module):
    def __init__(
        self,
        num_classes=4,
        beta=0.95,
        feature_threshold=1.0,
        output_threshold=1.0,
        image_height=64,
        image_width=64,
    ):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid(slope=25)

        self.conv1 = nn.Conv2d(1, 12, kernel_size=5, stride=2, padding=2)
        self.lif1 = snn.Leaky(
            beta=beta, threshold=feature_threshold, spike_grad=spike_grad
        )
        self.conv2 = nn.Conv2d(12, 24, kernel_size=5, stride=2, padding=2)
        self.lif2 = snn.Leaky(
            beta=beta, threshold=feature_threshold, spike_grad=spike_grad
        )

        flattened_size = 24 * (image_height // 4) * (image_width // 4)
        self.fc1 = nn.Linear(flattened_size, 128)
        self.lif3 = snn.Leaky(
            beta=beta, threshold=feature_threshold, spike_grad=spike_grad
        )
        self.fc_out = nn.Linear(128, num_classes)
        self.lif_out = snn.Leaky(
            beta=beta, threshold=output_threshold, spike_grad=spike_grad
        )

    def forward(self, spike_movie):
        num_steps = spike_movie.shape[1]
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem_out = self.lif_out.init_leaky()
        spikes, membranes = [], []

        for timestep in range(num_steps):
            current = self.conv1(spike_movie[:, timestep].unsqueeze(1))
            spike1, mem1 = self.lif1(current, mem1)
            current = self.conv2(spike1)
            spike2, mem2 = self.lif2(current, mem2)
            current = self.fc1(spike2.flatten(start_dim=1))
            spike3, mem3 = self.lif3(current, mem3)
            current = self.fc_out(spike3)
            spike_out, mem_out = self.lif_out(current, mem_out)
            spikes.append(spike_out)
            membranes.append(mem_out)

        return torch.stack(spikes), torch.stack(membranes)

In [7]:
def evaluate_classifier(model, data_loader, device=device, verbose=False):
    model.eval()
    labels_all, spike_predictions_all, membrane_predictions_all = [], [], []
    spike_counts_all = []

    with torch.no_grad():
        for spike_movies, labels_batch in data_loader:
            spike_movies = spike_movies.to(device)
            labels_batch = labels_batch.to(device)
            spk_rec, mem_rec = model(spike_movies)
            spike_counts = spk_rec.sum(dim=0)
            membrane_scores = mem_rec.sum(dim=0)
            labels_all.append(labels_batch.cpu())
            spike_predictions_all.append(spike_counts.argmax(dim=1).cpu())
            membrane_predictions_all.append(membrane_scores.argmax(dim=1).cpu())
            spike_counts_all.append(spike_counts.cpu())

    labels_all = torch.cat(labels_all)
    spike_predictions_all = torch.cat(spike_predictions_all)
    membrane_predictions_all = torch.cat(membrane_predictions_all)
    spike_counts_all = torch.cat(spike_counts_all)
    total_spikes = spike_counts_all.sum(dim=1)

    result = {
        "spike_accuracy": (spike_predictions_all == labels_all).float().mean().item(),
        "membrane_accuracy": (
            membrane_predictions_all == labels_all
        ).float().mean().item(),
        "zero_spike_samples": int((total_spikes == 0).sum().item()),
        "mean_output_spikes": total_spikes.float().mean().item(),
    }
    if verbose:
        print(pd.Series(result))
        print("Spike prediction counts:", torch.bincount(
            spike_predictions_all, minlength=num_classes
        ).tolist())
        print("Membrane prediction counts:", torch.bincount(
            membrane_predictions_all, minlength=num_classes
        ).tolist())
    return result


def conv_snn_accuracy(model, data_loader, device=device):
    return evaluate_classifier(model, data_loader, device)["spike_accuracy"]

In [8]:
def train_conv_spiking_classifier(
    train_loader,
    test_loader,
    num_classes=4,
    image_height=64,
    image_width=64,
    num_epochs=50,
    learning_rate=5e-4,
    beta=0.95,
    feature_threshold=1.0,
    output_threshold=1.0,
    device=device,
):
    model = ConvSpikingClassifier(
        num_classes=num_classes,
        beta=beta,
        feature_threshold=feature_threshold,
        output_threshold=output_threshold,
        image_height=image_height,
        image_width=image_width,
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.999),
        weight_decay=1e-5,
    )
    loss_fn = nn.CrossEntropyLoss()
    history = []
    best_test_accuracy = -1.0
    best_model_state = None

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0

        for spike_movies, labels_batch in train_loader:
            spike_movies = spike_movies.to(device)
            labels_batch = labels_batch.to(device)
            optimizer.zero_grad()
            _, mem_rec = model(spike_movies)

            # Current best objective: membrane cross-entropy at every timestep.
            loss = torch.zeros((), device=device)
            for timestep in range(mem_rec.shape[0]):
                loss = loss + loss_fn(mem_rec[timestep], labels_batch)

            loss.backward()
            optimizer.step()
            batch_size = labels_batch.shape[0]
            total_loss += loss.item() * batch_size
            total_examples += batch_size

        train_loss = total_loss / total_examples
        train_accuracy = conv_snn_accuracy(model, train_loader, device)
        test_accuracy = conv_snn_accuracy(model, test_loader, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
        })

        if test_accuracy > best_test_accuracy:
            best_test_accuracy = test_accuracy
            best_model_state = copy.deepcopy(model.state_dict())

        print(
            f"epoch {epoch:02d} | loss {train_loss:.4f} | "
            f"train acc {train_accuracy:.3f} | test acc {test_accuracy:.3f}"
        )

    model.load_state_dict(best_model_state)
    return model, pd.DataFrame(history)

In [9]:
SEEDS = [0, 1, 2, 3, 4]
NUM_EPOCHS = 50


def run_threshold_comparison(output_threshold, feature_threshold=1.0):
    rows = []
    models = {"raw": {}, "memory": {}}
    histories = {"raw": {}, "memory": {}}

    for seed in SEEDS:
        for input_name, train_loader, test_loader in [
            ("raw", raw_train_loader, raw_test_loader),
            ("memory", memory_train_loader, memory_test_loader),
        ]:
            print(
                f"\nSeed {seed} | {input_name} | "
                f"feature threshold={feature_threshold} | "
                f"output threshold={output_threshold}"
            )
            set_all_seeds(seed)
            model, history = train_conv_spiking_classifier(
                train_loader=train_loader,
                test_loader=test_loader,
                num_classes=num_classes,
                image_height=height,
                image_width=width,
                num_epochs=NUM_EPOCHS,
                learning_rate=5e-4,
                beta=0.95,
                feature_threshold=feature_threshold,
                output_threshold=output_threshold,
                device=device,
            )
            diagnostic = evaluate_classifier(model, test_loader, device)
            models[input_name][seed] = model
            histories[input_name][seed] = history
            rows.append({
                "seed": seed,
                "input": input_name,
                "feature_threshold": feature_threshold,
                "output_threshold": output_threshold,
                "best_test_accuracy": history["test_accuracy"].max(),
                "best_epoch": int(history["test_accuracy"].idxmax()),
                **diagnostic,
            })

    results = pd.DataFrame(rows)
    comparison = results.pivot(index="seed", columns="input", values="best_test_accuracy")
    comparison["memory_minus_raw"] = comparison["memory"] - comparison["raw"]
    display(results)
    display(comparison)
    display(comparison.mean().to_frame("mean"))
    return {"results": results, "comparison": comparison, "models": models, "histories": histories}

## Baseline: reproduce the current best pipeline

Run this cell to reproduce the original threshold-1.0 comparison. It performs ten training runs: raw and ObjectMemory for each of five seeds.

In [10]:
baseline_results = run_threshold_comparison(
    feature_threshold=1.0,
    output_threshold=1.0,
)


Seed 0 | raw | feature threshold=1.0 | output threshold=1.0
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 | 

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,1.0,1.0,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,1.0,1.0,0.2500,0,0.2500,0.2500,80,0.0000
2,1,raw,1.0,1.0,0.2500,0,0.2500,0.2500,80,0.0000
3,1,memory,1.0,1.0,0.5000,15,0.5000,0.4500,41,0.8875
4,2,raw,1.0,1.0,0.2500,0,0.2500,0.2500,80,0.0000
5,2,memory,1.0,1.0,0.6625,49,0.6625,0.6875,36,1.3750
6,3,raw,1.0,1.0,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,1.0,1.0,0.2500,0,0.2500,0.2500,80,0.0000
8,4,raw,1.0,1.0,0.4875,44,0.4875,0.5125,53,0.6875
9,4,memory,1.0,1.0,0.5500,25,0.5500,0.5625,9,2.1875


input,memory,raw,memory_minus_raw
seed,,,
0,0.2500,0.2500,0.0000
1,0.5000,0.2500,0.2500
2,0.6625,0.2500,0.4125
3,0.2500,0.2500,0.0000
4,0.5500,0.4875,0.0625


,mean
input,
memory,0.4425
raw,0.2975
memory_minus_raw,0.1450


## Next experiment A: output threshold 0.5

This keeps all hidden/feature thresholds at 1.0 and lowers only the four output neurons' threshold.

In [11]:
threshold_050_results = run_threshold_comparison(
    feature_threshold=1.0,
    output_threshold=0.5,
)


Seed 0 | raw | feature threshold=1.0 | output threshold=0.5
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 | 

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,1.0,0.5,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,1.0,0.5,0.2500,0,0.2500,0.2500,80,0.0000
2,1,raw,1.0,0.5,0.2500,0,0.2500,0.2500,80,0.0000
3,1,memory,1.0,0.5,0.4625,10,0.4625,0.3875,27,1.3000
4,2,raw,1.0,0.5,0.2500,0,0.2500,0.2500,80,0.0000
5,2,memory,1.0,0.5,0.7250,44,0.7250,0.6875,25,1.7750
6,3,raw,1.0,0.5,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,1.0,0.5,0.2500,0,0.2500,0.2500,80,0.0000
8,4,raw,1.0,0.5,0.5250,40,0.5250,0.5500,18,1.3625
9,4,memory,1.0,0.5,0.5500,20,0.5500,0.5375,0,3.2250


input,memory,raw,memory_minus_raw
seed,,,
0,0.2500,0.250,0.0000
1,0.4625,0.250,0.2125
2,0.7250,0.250,0.4750
3,0.2500,0.250,0.0000
4,0.5500,0.525,0.0250


,mean
input,
memory,0.4475
raw,0.3050
memory_minus_raw,0.1425


## Next experiment B: output threshold 0.75

In [12]:
threshold_075_results = run_threshold_comparison(
    feature_threshold=1.0,
    output_threshold=0.75,
)


Seed 0 | raw | feature threshold=1.0 | output threshold=0.75
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 |

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,1.0,0.75,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,1.0,0.75,0.2500,0,0.2500,0.2500,80,0.0000
2,1,raw,1.0,0.75,0.2500,0,0.2500,0.2500,80,0.0000
3,1,memory,1.0,0.75,0.5125,15,0.5125,0.4375,26,1.3375
4,2,raw,1.0,0.75,0.2500,0,0.2500,0.2500,80,0.0000
5,2,memory,1.0,0.75,0.6750,48,0.6750,0.6750,27,1.7375
6,3,raw,1.0,0.75,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,1.0,0.75,0.2500,0,0.2500,0.2500,80,0.0000
8,4,raw,1.0,0.75,0.5125,48,0.5125,0.5250,12,1.4125
9,4,memory,1.0,0.75,0.5750,21,0.5750,0.5375,7,2.3375


input,memory,raw,memory_minus_raw
seed,,,
0,0.2500,0.2500,0.0000
1,0.5125,0.2500,0.2625
2,0.6750,0.2500,0.4250
3,0.2500,0.2500,0.0000
4,0.5750,0.5125,0.0625


,mean
input,
memory,0.4525
raw,0.3025
memory_minus_raw,0.1500


## Optional combined summary

Run after completing the baseline and both new threshold experiments.

In [13]:
combined_results = pd.concat(
    [
        baseline_results["results"],
        threshold_050_results["results"],
        threshold_075_results["results"],
    ],
    ignore_index=True,
)

display(
    combined_results.groupby(["output_threshold", "input"])[
        [
            "best_test_accuracy",
            "spike_accuracy",
            "membrane_accuracy",
            "zero_spike_samples",
            "mean_output_spikes",
        ]
    ].agg(["mean", "std"])
)

best_test_accuracy           spike_accuracy            \
                                      mean       std           mean       std   
output_threshold input                                                          
0.50             memory             0.4475  0.203562         0.4475  0.203562   
                 raw                0.3050  0.122984         0.3050  0.122984   
0.75             memory             0.4525  0.193730         0.4525  0.193730   
                 raw                0.3025  0.117394         0.3025  0.117394   
1.00             memory             0.4425  0.185321         0.4425  0.185321   
                 raw                0.2975  0.106213         0.2975  0.106213   

                        membrane_accuracy           zero_spike_samples  \
                                     mean       std               mean   
output_threshold input                                                   
0.50             memory            0.4225  0.189860               42.4   
                 raw               0.3100  0.134164               67.6   
0.75             memory            0.4300  0.184687               44.0   
                 raw               0.3050  0.122984               66.4   
1.00             memory            0.4400  0.192719               49.2   
                 raw               0.3025  0.117394               74.6   

                                   mean_output_spikes            
                               std               mean       std  
output_threshold input                                           
0.50             memory  35.934663             1.2600  1.351226  
                 raw     27.727243             0.2725  0.609329  
0.75             memory  33.815677             1.0825  1.050320  
                 raw     30.410524             0.2825  0.631689  
1.00             memory  30.638211             0.8900  0.935807  
                 raw     12.074767             0.1375  0.307459

In [16]:
diagnostic_columns = [
    "seed",
    "input",
    "output_threshold",
    "best_test_accuracy",
    "spike_accuracy",
    "membrane_accuracy",
    "zero_spike_samples",
    "mean_output_spikes",
]

print("OUTPUT THRESHOLD = 0.50")
display(
    threshold_050_results["results"][diagnostic_columns]
    .sort_values(["seed", "input"])
    .reset_index(drop=True)
)

print("\nOUTPUT THRESHOLD = 0.75")
display(
    threshold_075_results["results"][diagnostic_columns]
    .sort_values(["seed", "input"])
    .reset_index(drop=True)
)

OUTPUT THRESHOLD = 0.50


,seed,input,output_threshold,best_test_accuracy,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,memory,0.5,0.2500,0.2500,0.2500,80,0.0000
1,0,raw,0.5,0.2500,0.2500,0.2500,80,0.0000
2,1,memory,0.5,0.4625,0.4625,0.3875,27,1.3000
3,1,raw,0.5,0.2500,0.2500,0.2500,80,0.0000
4,2,memory,0.5,0.7250,0.7250,0.6875,25,1.7750
5,2,raw,0.5,0.2500,0.2500,0.2500,80,0.0000
6,3,memory,0.5,0.2500,0.2500,0.2500,80,0.0000
7,3,raw,0.5,0.2500,0.2500,0.2500,80,0.0000
8,4,memory,0.5,0.5500,0.5500,0.5375,0,3.2250
9,4,raw,0.5,0.5250,0.5250,0.5500,18,1.3625



OUTPUT THRESHOLD = 0.75


,seed,input,output_threshold,best_test_accuracy,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,memory,0.75,0.2500,0.2500,0.2500,80,0.0000
1,0,raw,0.75,0.2500,0.2500,0.2500,80,0.0000
2,1,memory,0.75,0.5125,0.5125,0.4375,26,1.3375
3,1,raw,0.75,0.2500,0.2500,0.2500,80,0.0000
4,2,memory,0.75,0.6750,0.6750,0.6750,27,1.7375
5,2,raw,0.75,0.2500,0.2500,0.2500,80,0.0000
6,3,memory,0.75,0.2500,0.2500,0.2500,80,0.0000
7,3,raw,0.75,0.2500,0.2500,0.2500,80,0.0000
8,4,memory,0.75,0.5750,0.5750,0.5375,7,2.3375
9,4,raw,0.75,0.5125,0.5125,0.5250,12,1.4125


## Need to test upstream layers

In [17]:
@torch.no_grad()
def diagnose_layer_spiking(model, data_loader, device=device):
    """
    Measure where activity disappears inside the trained SNN.

    For each LIF layer:
      - mean total spikes per test sample across all timesteps
      - number of test samples with zero spikes across all timesteps
    """
    model.eval()

    layer_names = ["lif1", "lif2", "lif3", "lif_out"]

    # Stores per-sample total spike counts across the whole test set
    all_sample_counts = {name: [] for name in layer_names}

    # Temporary storage for one batch.
    # Each LIF module is called once per timestep.
    batch_counts = {name: [] for name in layer_names}

    hooks = []

    def make_hook(name):
        def hook(module, inputs, output):
            # snn.Leaky returns (spikes, membrane)
            spikes = output[0]

            # Sum over every dimension except batch.
            # Gives one spike count per sample for this timestep.
            per_sample = spikes.flatten(start_dim=1).sum(dim=1)

            batch_counts[name].append(per_sample.detach().cpu())

        return hook

    # Attach hooks
    for name in layer_names:
        layer = getattr(model, name)
        hooks.append(layer.register_forward_hook(make_hook(name)))

    for spike_movies, _ in data_loader:
        spike_movies = spike_movies.to(device)

        # Clear this batch's timestep records
        for name in layer_names:
            batch_counts[name] = []

        # Normal forward pass
        model(spike_movies)

        # Sum each sample's activity across all timesteps
        for name in layer_names:
            timestep_counts = torch.stack(batch_counts[name], dim=0)
            sample_totals = timestep_counts.sum(dim=0)
            all_sample_counts[name].append(sample_totals)

    # Remove hooks
    for hook in hooks:
        hook.remove()

    rows = []

    for name in layer_names:
        counts = torch.cat(all_sample_counts[name])

        rows.append({
            "layer": name,
            "mean_spikes_per_sample": counts.float().mean().item(),
            "zero_spike_samples": int((counts == 0).sum().item()),
            "zero_spike_fraction": float((counts == 0).float().mean().item()),
        })

    return pd.DataFrame(rows)

In [18]:
memory_seed2_050 = threshold_050_results["models"]["memory"][2]
memory_seed3_050 = threshold_050_results["models"]["memory"][3]

print("MEMORY — threshold 0.5 — seed 2 (successful)")
display(
    diagnose_layer_spiking(
        memory_seed2_050,
        memory_test_loader,
        device,
    )
)

print("MEMORY — threshold 0.5 — seed 3 (collapsed)")
display(
    diagnose_layer_spiking(
        memory_seed3_050,
        memory_test_loader,
        device,
    )
)

MEMORY — threshold 0.5 — seed 2 (successful)


,layer,mean_spikes_per_sample,zero_spike_samples,zero_spike_fraction
0,lif1,3186.399902,0,0.0000
1,lif2,26.637501,0,0.0000
2,lif3,20.687500,8,0.1000
3,lif_out,1.775000,25,0.3125


MEMORY — threshold 0.5 — seed 3 (collapsed)


,layer,mean_spikes_per_sample,zero_spike_samples,zero_spike_fraction
0,lif1,50.924999,0,0.0
1,lif2,0.000000,80,1.0
2,lif3,0.000000,80,1.0
3,lif_out,0.000000,80,1.0


SO activity dies much earlier in the bad seeds  - as early as LIF1. Will attempt to lower feature threshold

In [19]:
threshold_075_results = run_threshold_comparison(
    feature_threshold=0.75,
    output_threshold=0.75,
)


Seed 0 | raw | feature threshold=0.75 | output threshold=0.75
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,0.75,0.75,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,0.75,0.75,0.4250,14,0.4250,0.4250,7,2.7625
2,1,raw,0.75,0.75,0.5500,21,0.5500,0.5125,11,2.2375
3,1,memory,0.75,0.75,0.4625,4,0.4625,0.4250,47,0.6750
4,2,raw,0.75,0.75,0.5125,18,0.5125,0.5500,34,1.4000
5,2,memory,0.75,0.75,0.5375,7,0.5375,0.5250,2,2.2875
6,3,raw,0.75,0.75,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,0.75,0.75,0.4125,6,0.4125,0.3750,21,1.3875
8,4,raw,0.75,0.75,0.5250,18,0.5250,0.5000,2,2.8750
9,4,memory,0.75,0.75,0.5250,11,0.5250,0.5250,0,3.6250


input,memory,raw,memory_minus_raw
seed,,,
0,0.4250,0.2500,0.1750
1,0.4625,0.5500,-0.0875
2,0.5375,0.5125,0.0250
3,0.4125,0.2500,0.1625
4,0.5250,0.5250,0.0000


,mean
input,
memory,0.4725
raw,0.4175
memory_minus_raw,0.0550


In [21]:
threshold_050_results = run_threshold_comparison(
    feature_threshold=0.75,
    output_threshold=0.5,
)


Seed 0 | raw | feature threshold=0.75 | output threshold=0.5
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 |

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,0.75,0.5,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,0.75,0.5,0.4250,14,0.4250,0.4500,2,3.7000
2,1,raw,0.75,0.5,0.5500,38,0.5500,0.4875,4,2.8625
3,1,memory,0.75,0.5,0.4500,17,0.4500,0.4250,1,4.1875
4,2,raw,0.75,0.5,0.5375,24,0.5375,0.5125,4,3.2875
5,2,memory,0.75,0.5,0.5250,9,0.5250,0.5500,0,3.9625
6,3,raw,0.75,0.5,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,0.75,0.5,0.4125,6,0.4125,0.3625,11,2.1875
8,4,raw,0.75,0.5,0.5750,16,0.5750,0.5000,0,4.9125
9,4,memory,0.75,0.5,0.5250,8,0.5250,0.4875,0,3.5000


input,memory,raw,memory_minus_raw
seed,,,
0,0.4250,0.2500,0.1750
1,0.4500,0.5500,-0.1000
2,0.5250,0.5375,-0.0125
3,0.4125,0.2500,0.1625
4,0.5250,0.5750,-0.0500


,mean
input,
memory,0.4675
raw,0.4325
memory_minus_raw,0.0350


In [22]:
threshold_075_results = run_threshold_comparison(
    feature_threshold=0.85,
    output_threshold=0.5,
)


Seed 0 | raw | feature threshold=0.85 | output threshold=0.5
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 |

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,0.85,0.5,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,0.85,0.5,0.4375,25,0.4375,0.4375,2,2.9875
2,1,raw,0.85,0.5,0.5875,40,0.5875,0.6125,0,5.1000
3,1,memory,0.85,0.5,0.5250,6,0.5250,0.3750,9,1.6750
4,2,raw,0.85,0.5,0.5875,46,0.5875,0.5750,11,1.8875
5,2,memory,0.85,0.5,0.5750,19,0.5750,0.5625,0,3.6000
6,3,raw,0.85,0.5,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,0.85,0.5,0.4500,9,0.4500,0.3875,22,1.5500
8,4,raw,0.85,0.5,0.5750,10,0.5750,0.5000,0,3.5625
9,4,memory,0.85,0.5,0.5500,10,0.5500,0.5375,0,3.1500


input,memory,raw,memory_minus_raw
seed,,,
0,0.4375,0.2500,0.1875
1,0.5250,0.5875,-0.0625
2,0.5750,0.5875,-0.0125
3,0.4500,0.2500,0.2000
4,0.5500,0.5750,-0.0250


,mean
input,
memory,0.5075
raw,0.4500
memory_minus_raw,0.0575


In [24]:
threshold_075_results = run_threshold_comparison(
    feature_threshold=0.9,
    output_threshold=0.5,
)


Seed 0 | raw | feature threshold=0.9 | output threshold=0.5
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 | 

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,0.9,0.5,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,0.9,0.5,0.2500,0,0.2500,0.2500,80,0.0000
2,1,raw,0.9,0.5,0.6000,49,0.6000,0.5875,2,2.0000
3,1,memory,0.9,0.5,0.5000,8,0.5000,0.4875,9,1.9750
4,2,raw,0.9,0.5,0.2500,0,0.2500,0.2500,80,0.0000
5,2,memory,0.9,0.5,0.5375,18,0.5375,0.5750,6,2.6625
6,3,raw,0.9,0.5,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,0.9,0.5,0.4750,13,0.4750,0.4250,15,1.7000
8,4,raw,0.9,0.5,0.5500,22,0.5500,0.4750,0,4.1125
9,4,memory,0.9,0.5,0.5500,15,0.5500,0.5250,0,3.9125


input,memory,raw,memory_minus_raw
seed,,,
0,0.2500,0.25,0.0000
1,0.5000,0.60,-0.1000
2,0.5375,0.25,0.2875
3,0.4750,0.25,0.2250
4,0.5500,0.55,0.0000


,mean
input,
memory,0.4625
raw,0.3800
memory_minus_raw,0.0825


In [27]:
def quantify_velocity_estimation_on_classifier_dataset(
    raw_spikes_dataset,
    samples_per_shape=100,
    base_seed=0,
    max_speed=3,
    max_velocity=3,
    velocity_params=GOOD_VELOCITY_PARAMS,
):
    rows = []

    sample_index = 0

    for shape_name in SHAPE_NAMES:
        for _ in range(samples_per_shape):

            # Re-create the EXACT velocity sequence used when this sample
            # was originally generated.
            true_velocity_sequence = sample_random_velocity_sequence(
                num_transitions=STEPS - 1,
                max_speed=max_speed,
                allow_stationary=False,
                seed=base_seed + sample_index,
            )

            movie = raw_spikes_dataset[sample_index]

            for transition_index, (true_dy, true_dx) in enumerate(
                true_velocity_sequence
            ):
                previous_t = transition_index
                current_t = transition_index + 1

                result = (
                    lu.estimate_velocity_with_explicit_motion_pairs_and_inhibition(
                        previous_spikes=movie[previous_t],
                        current_spikes=movie[current_t],
                        max_offset=max_velocity,
                        **velocity_params,
                    )
                )

                estimated_dy = result["estimated_dy"]
                estimated_dx = result["estimated_dx"]

                # This is the gate ACTUALLY used by the current
                # displacement/ObjectMemory pipeline:
                # all velocity neurons that spiked at least once.
                velocity_gate = lu.get_velocity_gate_from_result(
                    result,
                    use_total_spikes=True,
                )

                true_i = true_dy + max_velocity
                true_j = true_dx + max_velocity

                true_velocity_in_gate = bool(
                    velocity_gate[true_i, true_j].item() > 0
                )

                num_active_velocity_gates = int(
                    velocity_gate.sum().item()
                )

                rows.append({
                    "sample": sample_index,
                    "shape": shape_name,
                    "transition": f"{previous_t}->{current_t}",

                    "true_dy": true_dy,
                    "true_dx": true_dx,

                    "estimated_dy": estimated_dy,
                    "estimated_dx": estimated_dx,

                    "winner_correct": (
                        estimated_dy == true_dy
                        and estimated_dx == true_dx
                    ),

                    "correct_dy": estimated_dy == true_dy,
                    "correct_dx": estimated_dx == true_dx,

                    "true_velocity_in_gate": true_velocity_in_gate,
                    "num_active_velocity_gates": num_active_velocity_gates,

                    # Useful measure of how "slow" the motion is.
                    # 1 means at least one component has magnitude 1, etc.
                    "max_component_speed": max(
                        abs(true_dy),
                        abs(true_dx),
                    ),

                    "euclidean_speed": (
                        true_dy**2 + true_dx**2
                    ) ** 0.5,
                })

            sample_index += 1

            if sample_index % 50 == 0:
                print(f"Processed {sample_index}/400 movies")

    return pd.DataFrame(rows)


velocity_dataset_results = quantify_velocity_estimation_on_classifier_dataset(
    raw_spikes_dataset
)

print("Total transitions:", len(velocity_dataset_results))
display(velocity_dataset_results.head())

Processed 50/400 movies
Processed 100/400 movies
Processed 150/400 movies
Processed 200/400 movies
Processed 250/400 movies
Processed 300/400 movies
Processed 350/400 movies
Processed 400/400 movies
Total transitions: 2400


,sample,shape,transition,true_dy,true_dx,estimated_dy,estimated_dx,winner_correct,correct_dy,correct_dx,true_velocity_in_gate,num_active_velocity_gates,max_component_speed,euclidean_speed
0,0,seven,0->1,1,0,1,0,True,True,True,True,1,1,1.000000
1,0,seven,1->2,-3,-1,-3,-1,True,True,True,True,1,3,3.162278
2,0,seven,2->3,-2,-1,-2,-1,True,True,True,True,1,2,2.236068
3,0,seven,3->4,3,0,3,0,True,True,True,True,1,3,3.000000
4,0,seven,4->5,1,2,1,2,True,True,True,True,1,2,2.236068


In [28]:
print("OVERALL")
display(pd.Series({
    "total_transitions": len(velocity_dataset_results),
    "winner_accuracy": velocity_dataset_results["winner_correct"].mean(),
    "true_velocity_in_gate_rate": velocity_dataset_results["true_velocity_in_gate"].mean(),
    "mean_active_velocity_gates": velocity_dataset_results["num_active_velocity_gates"].mean(),
}))

print("\nBY MAX COMPONENT SPEED")
display(
    velocity_dataset_results
    .groupby("max_component_speed")
    .agg(
        transitions=("winner_correct", "size"),
        winner_accuracy=("winner_correct", "mean"),
        true_velocity_in_gate_rate=("true_velocity_in_gate", "mean"),
        mean_active_velocity_gates=("num_active_velocity_gates", "mean"),
    )
)

print("\nBY SHAPE")
display(
    velocity_dataset_results
    .groupby("shape")
    .agg(
        transitions=("winner_correct", "size"),
        winner_accuracy=("winner_correct", "mean"),
        true_velocity_in_gate_rate=("true_velocity_in_gate", "mean"),
        mean_active_velocity_gates=("num_active_velocity_gates", "mean"),
    )
)

OVERALL


total_transitions             2400.000000
winner_accuracy                  1.000000
true_velocity_in_gate_rate       1.000000
mean_active_velocity_gates       2.999167
dtype: float64


BY MAX COMPONENT SPEED


,transitions,winner_accuracy,true_velocity_in_gate_rate,mean_active_velocity_gates
max_component_speed,,,,
1,387,1.0,1.0,3.413437
2,811,1.0,1.0,3.102343
3,1202,1.0,1.0,2.796173



BY SHAPE


,transitions,winner_accuracy,true_velocity_in_gate_rate,mean_active_velocity_gates
shape,,,,
box,600,1.0,1.0,8.996667
ell,600,1.0,1.0,1.000000
seven,600,1.0,1.0,1.000000
tee,600,1.0,1.0,1.000000


## Would like to pivot into less overlap motion modes to test memory vs raw inputs

In [29]:
def sample_fast_velocity_sequence(
    num_transitions,
    min_component_speed=4,
    max_speed=6,
    seed=None,
):
    """
    Sample velocity vectors with sufficiently large motion.

    Requirement:
        max(|dy|, |dx|) >= min_component_speed

    so every transition moves substantially in at least one axis.
    """
    if seed is not None:
        torch.manual_seed(seed)

    velocities = []

    for _ in range(num_transitions):
        while True:
            dy = int(
                torch.randint(
                    -max_speed,
                    max_speed + 1,
                    (1,),
                ).item()
            )
            dx = int(
                torch.randint(
                    -max_speed,
                    max_speed + 1,
                    (1,),
                ).item()
            )

            if max(abs(dy), abs(dx)) >= min_component_speed:
                break

        velocities.append((dy, dx))

    return velocities

In [30]:
def make_one_fast_classifier_dataset_sample(
    shape_name,
    min_component_speed=4,
    max_speed=6,
    sample_seed=0,
    max_velocity=6,
    max_displacement=36,
    velocity_params=None,
):
    if velocity_params is None:
        velocity_params = GOOD_VELOCITY_PARAMS

    velocity_sequence = sample_fast_velocity_sequence(
        STEPS - 1,
        min_component_speed=min_component_speed,
        max_speed=max_speed,
        seed=sample_seed,
    )

    start_top, start_left = (
        choose_safe_start_position_for_velocity_sequence(
            shape_name,
            velocity_sequence,
            seed=sample_seed + 10_000,
        )
    )

    case = lu.make_variable_velocity_spike_case(
        velocity_sequence=velocity_sequence,
        shape_name=shape_name,
        start_top=start_top,
        start_left=start_left,
    )

    memory_result = lu.run_displacement_gated_object_memory(
        case=case,
        max_velocity=max_velocity,
        max_displacement=max_displacement,
        velocity_params=velocity_params,
        threshold_memory=1.5,
        beta_memory=0.8,
        reset_mode="subtract",
    )

    return {
        "raw_spikes": case["spikes"].float(),
        "object_memory_spikes":
            memory_result["memory_spike_history"].float(),
        "label": SHAPE_TO_LABEL[shape_name],

        # Save these now so diagnostics are much easier.
        "velocity_sequence": velocity_sequence,
        "start_top": start_top,
        "start_left": start_left,
    }

In [31]:
def build_fast_classifier_dataset(
    samples_per_shape=100,
    min_component_speed=4,
    max_speed=6,
    max_velocity=6,
    max_displacement=36,
    base_seed=0,
):
    raw_samples = []
    memory_samples = []
    labels = []
    velocity_sequences = []

    total = samples_per_shape * len(SHAPE_NAMES)
    sample_index = 0

    for shape_name in SHAPE_NAMES:
        shape_count = 0

        while shape_count < samples_per_shape:
            seed = base_seed + sample_index

            try:
                sample = make_one_fast_classifier_dataset_sample(
                    shape_name=shape_name,
                    min_component_speed=min_component_speed,
                    max_speed=max_speed,
                    sample_seed=seed,
                    max_velocity=max_velocity,
                    max_displacement=max_displacement,
                )

            except ValueError:
                # Trajectory could not fit safely on 64x64.
                # Try a new random sequence.
                sample_index += 1
                continue

            raw_samples.append(sample["raw_spikes"])
            memory_samples.append(
                sample["object_memory_spikes"]
            )
            labels.append(sample["label"])
            velocity_sequences.append(
                sample["velocity_sequence"]
            )

            shape_count += 1
            sample_index += 1

            if len(labels) % 25 == 0:
                print(
                    f"Generated {len(labels)}/{total} samples"
                )

    return {
        "raw_spikes_dataset":
            torch.stack(raw_samples),

        "object_memory_spikes_dataset":
            torch.stack(memory_samples),

        "labels":
            torch.tensor(labels, dtype=torch.long),

        "velocity_sequences":
            velocity_sequences,
    }

In [32]:
fast_classifier_dataset = build_fast_classifier_dataset(
    samples_per_shape=100,
    min_component_speed=4,
    max_speed=6,
    max_velocity=6,
    max_displacement=36,
    base_seed=0,
)

fast_raw_spikes_dataset = (
    fast_classifier_dataset["raw_spikes_dataset"]
)

fast_memory_spikes_dataset = (
    fast_classifier_dataset["object_memory_spikes_dataset"]
)

fast_labels = fast_classifier_dataset["labels"]

fast_velocity_sequences = (
    fast_classifier_dataset["velocity_sequences"]
)

print("Raw:", fast_raw_spikes_dataset.shape)
print("Memory:", fast_memory_spikes_dataset.shape)
print("Labels:", fast_labels.shape)
print("Class counts:", torch.bincount(fast_labels))

Generated 25/400 samples
Generated 50/400 samples
Generated 75/400 samples
Generated 100/400 samples
Generated 125/400 samples
Generated 150/400 samples
Generated 175/400 samples
Generated 200/400 samples
Generated 225/400 samples
Generated 250/400 samples
Generated 275/400 samples
Generated 300/400 samples
Generated 325/400 samples
Generated 350/400 samples
Generated 375/400 samples
Generated 400/400 samples
Raw: torch.Size([400, 7, 64, 64])
Memory: torch.Size([400, 7, 64, 64])
Labels: torch.Size([400])
Class counts: tensor([100, 100, 100, 100])


In [33]:
def mean_consecutive_jaccard(movie_dataset):
    values = []

    for movie in movie_dataset:
        movie = movie > 0

        for t in range(movie.shape[0] - 1):
            a = movie[t]
            b = movie[t + 1]

            intersection = (a & b).sum().item()
            union = (a | b).sum().item()

            if union > 0:
                values.append(intersection / union)

    return sum(values) / len(values)


print(
    "Mean RAW consecutive overlap:",
    mean_consecutive_jaccard(
        fast_raw_spikes_dataset
    )
)

print(
    "Mean MEMORY consecutive overlap:",
    mean_consecutive_jaccard(
        fast_memory_spikes_dataset
    )
)

Mean RAW consecutive overlap: 0.011965234769390785
Mean MEMORY consecutive overlap: 1.0


In [38]:
fast_train_indices, fast_test_indices = make_stratified_train_test_split(
    fast_labels,
    seed=0,
)

fast_raw_train_loader, fast_raw_test_loader = make_dataloaders(
    fast_raw_spikes_dataset,
    fast_labels,
    fast_train_indices,
    fast_test_indices,
)

fast_memory_train_loader, fast_memory_test_loader = make_dataloaders(
    fast_memory_spikes_dataset,
    fast_labels,
    fast_train_indices,
    fast_test_indices,
)

print("Train samples:", len(fast_train_indices))
print("Test samples:", len(fast_test_indices))
print(
    "Test class counts:",
    torch.bincount(fast_labels[fast_test_indices]).tolist()
)

Train samples: 320
Test samples: 80
Test class counts: [20, 20, 20, 20]


In [39]:
FAST_SEEDS = [0, 1, 2, 3, 4]
FAST_NUM_EPOCHS = 50


def run_fast_threshold_comparison(
    output_threshold,
    feature_threshold=0.75,
):
    rows = []

    models = {
        "raw": {},
        "memory": {},
    }

    histories = {
        "raw": {},
        "memory": {},
    }

    for seed in FAST_SEEDS:

        for input_name, train_loader, test_loader in [

            (
                "raw",
                fast_raw_train_loader,
                fast_raw_test_loader,
            ),

            (
                "memory",
                fast_memory_train_loader,
                fast_memory_test_loader,
            ),

        ]:

            print(
                f"\nSeed {seed} | {input_name} | "
                f"feature threshold={feature_threshold} | "
                f"output threshold={output_threshold}"
            )

            set_all_seeds(seed)

            model, history = train_conv_spiking_classifier(
                train_loader=train_loader,
                test_loader=test_loader,
                num_classes=4,
                image_height=64,
                image_width=64,
                num_epochs=FAST_NUM_EPOCHS,
                learning_rate=5e-4,
                beta=0.95,
                feature_threshold=feature_threshold,
                output_threshold=output_threshold,
                device=device,
            )

            diagnostic = evaluate_classifier(
                model,
                test_loader,
                device=device,
            )

            models[input_name][seed] = model
            histories[input_name][seed] = history

            rows.append({
                "seed": seed,
                "input": input_name,

                "feature_threshold":
                    feature_threshold,

                "output_threshold":
                    output_threshold,

                "best_test_accuracy":
                    history["test_accuracy"].max(),

                "best_epoch":
                    int(
                        history["test_accuracy"].idxmax()
                    ),

                **diagnostic,
            })

    results = pd.DataFrame(rows)

    comparison = results.pivot(
        index="seed",
        columns="input",
        values="best_test_accuracy",
    )

    comparison["memory_minus_raw"] = (
        comparison["memory"]
        - comparison["raw"]
    )

    print("\nFULL RESULTS")
    display(results)

    print("\nMEMORY VS RAW BY SEED")
    display(comparison)

    print("\nMEAN RESULTS")
    display(
        comparison.mean().to_frame("mean")
    )

    return {
        "results": results,
        "comparison": comparison,
        "models": models,
        "histories": histories,
    }

In [40]:
fast_results = run_fast_threshold_comparison(
    feature_threshold=0.75,
    output_threshold=0.75,
)


Seed 0 | raw | feature threshold=0.75 | output threshold=0.75
epoch 00 | loss 9.8206 | train acc 0.250 | test acc 0.250
epoch 01 | loss 9.8057 | train acc 0.250 | test acc 0.250
epoch 02 | loss 9.7950 | train acc 0.250 | test acc 0.250
epoch 03 | loss 9.7814 | train acc 0.250 | test acc 0.250
epoch 04 | loss 9.7720 | train acc 0.250 | test acc 0.250
epoch 05 | loss 9.7640 | train acc 0.250 | test acc 0.250
epoch 06 | loss 9.7566 | train acc 0.250 | test acc 0.250
epoch 07 | loss 9.7491 | train acc 0.250 | test acc 0.250
epoch 08 | loss 9.7431 | train acc 0.250 | test acc 0.250
epoch 09 | loss 9.7384 | train acc 0.250 | test acc 0.250
epoch 10 | loss 9.7339 | train acc 0.250 | test acc 0.250
epoch 11 | loss 9.7305 | train acc 0.250 | test acc 0.250
epoch 12 | loss 9.7285 | train acc 0.250 | test acc 0.250
epoch 13 | loss 9.7242 | train acc 0.250 | test acc 0.250
epoch 14 | loss 9.7210 | train acc 0.250 | test acc 0.250
epoch 15 | loss 9.7194 | train acc 0.250 | test acc 0.250
epoch 16 

,seed,input,feature_threshold,output_threshold,best_test_accuracy,best_epoch,spike_accuracy,membrane_accuracy,zero_spike_samples,mean_output_spikes
0,0,raw,0.75,0.75,0.2500,0,0.2500,0.2500,80,0.0000
1,0,memory,0.75,0.75,0.3875,31,0.3875,0.3500,3,3.9000
2,1,raw,0.75,0.75,0.5125,24,0.5125,0.5375,47,1.6750
3,1,memory,0.75,0.75,0.5125,12,0.5125,0.5000,1,3.1625
4,2,raw,0.75,0.75,0.5875,49,0.5875,0.6500,3,2.8875
5,2,memory,0.75,0.75,0.5250,38,0.5250,0.4750,0,5.1375
6,3,raw,0.75,0.75,0.2500,0,0.2500,0.2500,80,0.0000
7,3,memory,0.75,0.75,0.4500,25,0.4500,0.4375,1,3.5750
8,4,raw,0.75,0.75,0.6000,15,0.6000,0.5500,0,3.7750
9,4,memory,0.75,0.75,0.5125,10,0.5125,0.4875,1,3.4000



MEMORY VS RAW BY SEED


input,memory,raw,memory_minus_raw
seed,,,
0,0.3875,0.2500,0.1375
1,0.5125,0.5125,0.0000
2,0.5250,0.5875,-0.0625
3,0.4500,0.2500,0.2000
4,0.5125,0.6000,-0.0875



MEAN RESULTS


,mean
input,
memory,0.4775
raw,0.4400
memory_minus_raw,0.0375
